# Audit provenance — every number, and the cell that produced it

## What this notebook is for

The audit document states a lot of figures. This notebook produces **all of them**,
one labelled cell per claim, so any of them can be cross-checked by running the cell
rather than taking the document's word for it.

Each cell prints a tag like `[K1]`. The same tag appears beside the figure in the
document. If a number in the document has no tag, it is not evidence — treat it as
unverified.

**Nothing here modifies the pipeline.** It only reads from `sprint_pipeline.py`.

In [1]:
from project_paths import PATHS

import numpy as np, pandas as pd, scipy.stats as st
from scipy.signal import find_peaks
import sprint_pipeline as SP

N = SP.ANGLE_NAMES
JOINTS = ["hip", "knee", "ankle", "shoulder", "elbow"]

def kneedle(t, v):
    """Velocity-curve inflection: max perpendicular distance from the endpoint chord."""
    x = (t - t[0]) / (t[-1] - t[0]); y = (v - v[0]) / (v.max() - v[0])
    return int(np.argmax(y - x))

def hip_ext_peak(cycle):
    """Peak hip extension, degrees the thigh travels BEHIND the trunk line.

    Computed here rather than imported: `hip_extension_peak` was removed from
    sprint_pipeline.py in a refactor outside this analysis, so the audit defines
    it locally to stay reproducible against the code as it now stands.
    """
    lo = [cycle[:, N.index(nm)].min() for nm in ("hip_R", "hip_L")]
    return float(-np.mean(lo))

def partial(x, y, z):
    """Correlation of x and y with z linearly removed from both."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    return st.pearsonr(rx, ry)

def icc11(df, col, group="pid"):
    """Shrout-Fleiss ICC(1,1) by one-way ANOVA."""
    g = df.groupby(group)[col]; k = g.size().mean(); n = g.ngroups
    MSB = k * ((g.mean() - df[col].mean()) ** 2).sum() / (n - 1)
    MSW = sum(((x - x.mean()) ** 2).sum() for _, x in g) / (len(df) - n)
    return (MSB - MSW) / (MSB + (k - 1) * MSW)

print("helpers ready")

helpers ready


## Load the cohort once

One pass over the C3D files. Everything below reuses this.

In [2]:
cyc, raw, stance, meta, per_stride = {}, {}, {}, {}, []

for f in sorted(SP.C3D_DIR.glob("*.c3d")):
    pid = f.stem.split("-")[0].strip()
    if pid in SP.EXCLUDED_PIDS:
        continue
    tr = SP.clean_for_angles(f); fs = tr["fs"]; r64 = tr["raw64"]
    cy, _n, _sp = SP.stride_cycle_angles(r64, tr["mk"], tr["peak_frame"])
    if cy is None:
        continue

    cyc[pid] = cy
    raw[pid] = SP.stride_cycle_raw64(r64, tr["mk"], tr["peak_frame"])
    ang, _ = SP.compute_joint_angles(r64)
    w, _ = SP.find_top_speed_strides(tr["mk"], tr["peak_frame"])

    # ground contact: right foot within 3 cm of its own local ground level
    z = np.minimum(r64[:, SP.MK_R_HEEL, SP._VERT], r64[:, SP.MK_R_TOE, SP._VERT])
    g = np.percentile(z, 2)
    stance[pid] = np.mean([SP.resample_to_cycle((z[a:b+1] < g + 0.03).astype(float))
                           for a, b in w], axis=0) > 0.5

    vel = SP.smooth_velocity_curve(SP.compute_velocity(tr["mk"], fs, raw64=r64)[0])
    meta[pid] = dict(
        v      = float(vel.max()),
        v_raw  = tr["peak_vel_ms"],
        h      = SP.PARTICIPANT_ANTHRO[pid]["body_height"],
        gct    = np.mean([(z[a:b+1] < g + 0.03).sum() / fs for a, b in w]),
        stride = np.mean([(b - a) / fs for a, b in w]),
        frames = np.mean([b - a for a, b in w]),
        elbow_t= kneedle(np.arange(len(vel)) / fs, vel) / fs,
        elbow_pct = float(vel[kneedle(np.arange(len(vel)) / fs, vel)] / vel.max() * 100),
    )
    meta[pid]["flight"] = meta[pid]["stride"] - meta[pid]["gct"]

    for a, b in w:                       # one row per individual stride
        c = np.column_stack([SP.resample_to_cycle(ang[a:b+1, j]) for j in range(ang.shape[1])])
        per_stride.append(dict(pid=pid, hip_ext_peak=hip_ext_peak(c),
                               **SP.scalar_features(c)))

pids = sorted(cyc)
M    = pd.DataFrame(meta).T.astype(float).loc[pids]
PS   = pd.DataFrame(per_stride)
v, h, gct = M.v.values, M.h.values, M.gct.values

print(f"{len(pids)} athletes | {len(PS)} individual strides | velocity "
      f"{v.min():.2f}-{v.max():.2f} m/s")

30 athletes | 150 individual strides | velocity 6.39-9.35 m/s


---
# P · What a principal component actually is

## [P1] The input shape — the feature count is 1313, not 13

In [3]:
X = np.stack([cyc[p] for p in pids])
print(f"[P1] X.shape = {X.shape}  = (athletes, cycle points, angles)")
print(f"[P1] features fed to the PCA = 101 x 13 = {101*13}")
print(f"[P1] ANGLE_NAMES ({len(N)}) = {N}")

[P1] X.shape = (30, 101, 13)  = (athletes, cycle points, angles)
[P1] features fed to the PCA = 101 x 13 = 1313
[P1] ANGLE_NAMES (13) = ['hip_R', 'hip_L', 'knee_R', 'knee_L', 'ankle_R', 'ankle_L', 'shoulder_R', 'shoulder_L', 'elbow_R', 'elbow_L', 'trunk_lean', 'thigh_separation', 'arm_separation']


## [P2] Standardisation, flattening, and the SVD

In [4]:
Xs = SP.smooth_angle_curves(X, penalty=1.0)
sd = Xs.std(axis=(0, 1), keepdims=True)
print("[P2] per-angle SD used as the divisor (degrees):")
for n_, s_ in zip(N, sd.ravel()):
    print(f"       {n_:18} {s_:6.1f}")

flat = (Xs / sd).reshape(len(pids), -1)
print(f"\n[P2] flattened -> {flat.shape}: each athlete is ONE row of {flat.shape[1]} numbers")
print(f"[P2] column j holds cycle point j//13, angle j%13")

U, S, Vt = np.linalg.svd(flat - flat.mean(0), full_matrices=False)
print(f"[P2] SVD: U{U.shape}  S{S.shape}  Vt{Vt.shape}")
print(f"[P2] rank cap = n-1 = {len(pids)-1}, NOT the 1313 features")

[P2] per-angle SD used as the divisor (degrees):
       hip_R                24.1
       hip_L                24.1
       knee_R               38.8
       knee_L               38.3
       ankle_R              18.3
       ankle_L              18.3
       shoulder_R           29.9
       shoulder_L           29.8
       elbow_R              29.5
       elbow_L              29.6
       trunk_lean            4.6
       thigh_separation     45.7
       arm_separation       58.2

[P2] flattened -> (30, 1313): each athlete is ONE row of 1313 numbers
[P2] column j holds cycle point j//13, angle j%13
[P2] SVD: U(30, 30)  S(30,)  Vt(30, 1313)
[P2] rank cap = n-1 = 29, NOT the 1313 features


## [P3] PC1 itself — shape, head, and unit length

In [5]:
pc1 = Vt[0]
print(f"[P3] pc1.shape = {pc1.shape}   reshaped = {pc1.reshape(101,13).shape} (points, angles)")
print(f"[P3] sum of squares = {(pc1**2).sum():.6f}  (unit vector, so shares are exact)")
print("\n[P3] HEAD of PC1 — the first 13 entries are cycle point 0, all 13 angles:")
for j in range(13):
    print(f"       [{j:2d}] {N[j]:18} {pc1[j]:+.5f}")
print(f"\n[P3] entries 13-25 are cycle point 1, same 13 angles:")
print(f"       {np.round(pc1[13:26], 5)}")

[P3] pc1.shape = (1313,)   reshaped = (101, 13) (points, angles)
[P3] sum of squares = 1.000000  (unit vector, so shares are exact)

[P3] HEAD of PC1 — the first 13 entries are cycle point 0, all 13 angles:
       [ 0] hip_R              -0.01656
       [ 1] hip_L              -0.03344
       [ 2] knee_R             -0.00337
       [ 3] knee_L             -0.00731
       [ 4] ankle_R            -0.00827
       [ 5] ankle_L            -0.00556
       [ 6] shoulder_R         -0.00936
       [ 7] shoulder_L         +0.00569
       [ 8] elbow_R            -0.00333
       [ 9] elbow_L            +0.00797
       [10] trunk_lean         -0.08309
       [11] thigh_separation   +0.00891
       [12] arm_separation     -0.00771

[P3] entries 13-25 are cycle point 1, same 13 angles:
       [-0.0157  -0.03248 -0.00334 -0.00636 -0.00683 -0.00544 -0.0094   0.00587
 -0.00407  0.00811 -0.07929  0.00886 -0.00783]


## [P4] Where PC1's length sits, per angle

In [6]:
L = pc1.reshape(101, 13); tot = (L**2).sum()
print("[P4] share of PC1's squared length, by angle:")
for j in np.argsort(-(L**2).sum(0)):
    print(f"       {N[j]:18} {(L[:,j]**2).sum()/tot*100:5.1f} %")
print(f"\n[P4] trunk_lean loading across the cycle (every 10th point):")
print(f"       {np.round(L[::10,10],4)}   <- nearly flat: a postural offset, not a timing pattern")

[P4] share of PC1's squared length, by angle:
       trunk_lean          77.1 %
       hip_R                7.7 %
       hip_L                6.8 %
       elbow_L              2.5 %
       shoulder_L           1.8 %
       elbow_R              0.8 %
       knee_L               0.7 %
       arm_separation       0.6 %
       thigh_separation     0.5 %
       knee_R               0.5 %
       shoulder_R           0.5 %
       ankle_R              0.2 %
       ankle_L              0.2 %

[P4] trunk_lean loading across the cycle (every 10th point):
       [-0.0831 -0.067  -0.0788 -0.0977 -0.0979 -0.0844 -0.0758 -0.0806 -0.0968
 -0.104  -0.0837]   <- nearly flat: a postural offset, not a timing pattern


## [P5] What each component's scores track, and how it correlates with speed

In [7]:
fp = SP.run_angle_fpca(dict(zip(pids, Xs)), n_components=6)
cand = {}
for j, n_ in enumerate(N):
    Ccol = np.array([cyc[p][:, j] for p in pids])
    cand[f"{n_}.ROM"]  = np.ptp(Ccol, 1); cand[f"{n_}.mean"] = Ccol.mean(1)
    cand[f"{n_}.min"]  = Ccol.min(1);     cand[f"{n_}.max"]  = Ccol.max(1)
cand["contact time"] = gct; cand["body height"] = h; cand["stride freq"] = 1/M.stride.values

for k in range(6):
    sc = fp["scores"][:, k]; Lk = fp["loadings"][k]; tk = (Lk**2).sum()
    top = sorted(((Lk[:,j]**2).sum()/tk*100, N[j]) for j in range(13))[::-1][:2]
    cor = sorted(((abs(np.corrcoef(sc,x)[0,1]), np.corrcoef(sc,x)[0,1], nm)
                  for nm, x in cand.items()), reverse=True)[:2]
    print(f"[P5] PC{k+1}  {fp['explained_var'][k]:4.1f}%  r(score,velocity) = "
          f"{np.corrcoef(sc, v)[0,1]:+.3f}")
    print(f"       loading: " + ", ".join(f"{n_} {s_:.0f}%" for s_, n_ in top))
    print(f"       tracks : " + ", ".join(f"{nm} {r_:+.2f}" for _, r_, nm in cor))

[P5] PC1  42.0%  r(score,velocity) = -0.177
       loading: trunk_lean 77%, hip_R 8%
       tracks : trunk_lean.mean -0.98, trunk_lean.min -0.96
[P5] PC2  15.1%  r(score,velocity) = -0.175
       loading: ankle_L 16%, ankle_R 16%
       tracks : ankle_R.mean +0.71, ankle_L.mean +0.57
[P5] PC3   8.3%  r(score,velocity) = -0.107
       loading: ankle_L 49%, elbow_R 13%
       tracks : ankle_L.mean -0.72, ankle_L.min -0.64
[P5] PC4   6.6%  r(score,velocity) = +0.206
       loading: elbow_L 41%, elbow_R 38%
       tracks : elbow_R.min -0.69, elbow_R.mean -0.66
[P5] PC5   5.0%  r(score,velocity) = +0.022
       loading: elbow_L 39%, elbow_R 16%
       tracks : elbow_L.max +0.81, elbow_L.ROM +0.65
[P5] PC6   3.4%  r(score,velocity) = +0.124
       loading: ankle_R 26%, elbow_L 18%
       tracks : ankle_R.min +0.39, shoulder_R.max -0.38


## [P6] Why no component finds the knee

In [8]:
print("[P6] knee ROM and contact time against every component:")
kneeROM = np.ptp(np.array([cyc[p][:, N.index('knee_R')] for p in pids]), 1)
for k in range(6):
    sc = fp["scores"][:, k]
    print(f"       PC{k+1}: r(score, knee_R ROM) = {np.corrcoef(sc,kneeROM)[0,1]:+.3f}   "
          f"r(score, contact time) = {np.corrcoef(sc,gct)[0,1]:+.3f}")

print("\n[P6] between-athlete vs within-cycle spread — why the PCA weights what it does:")
print(f"       {'angle':18}{'within-cycle SD':>17}{'between-athlete SD':>20}{'ratio':>8}")
for n_ in ["trunk_lean", "ankle_L", "hip_R", "knee_R"]:
    Ccol = np.array([cyc[p][:, N.index(n_)] for p in pids])
    w_, b_ = Ccol.std(axis=1).mean(), Ccol.mean(axis=1).std()
    print(f"       {n_:18}{w_:17.1f}{b_:20.1f}{b_/w_:8.2f}")

[P6] knee ROM and contact time against every component:
       PC1: r(score, knee_R ROM) = +0.096   r(score, contact time) = +0.184
       PC2: r(score, knee_R ROM) = +0.053   r(score, contact time) = +0.232
       PC3: r(score, knee_R ROM) = +0.124   r(score, contact time) = -0.075
       PC4: r(score, knee_R ROM) = -0.213   r(score, contact time) = -0.372
       PC5: r(score, knee_R ROM) = +0.074   r(score, contact time) = -0.077
       PC6: r(score, knee_R ROM) = -0.176   r(score, contact time) = -0.179

[P6] between-athlete vs within-cycle spread — why the PCA weights what it does:
       angle               within-cycle SD  between-athlete SD   ratio
       trunk_lean                      2.4                 4.1    1.72
       ankle_L                        18.1                 6.9    0.38
       hip_R                          23.2                 7.5    0.32
       knee_R                         39.3                 4.6    0.12


---
# K · What "the knee correlates with speed" is

## [K1] The ISB sign convention, after correction

In [9]:
K = np.array([cyc[p][:, N.index("knee_R")] for p in pids])
print(f"[K1] knee_R spans {K.min(1).mean():.1f} to {K.max(1).mean():.1f} deg")
print("[K1] ISB (Wu 2002, after Grood & Suntay 1983): flexion POSITIVE, 0 = full extension")
print(f"[K1] peak flexion measured here = {K.max(1).mean():.1f} deg")
print("[K1] published (Miyashiro & Nagahara 2019, n=79, 40-50 m of a 60 m sprint):")
print("       minimum knee angle in swing 31.6 +/- 5.6 deg on a 180=straight scale")
print(f"       -> equivalent peak flexion 148.4 deg; ours reads {148.4-K.max(1).mean():.1f} deg lower")

[K1] knee_R spans 16.9 to 131.8 deg
[K1] ISB (Wu 2002, after Grood & Suntay 1983): flexion POSITIVE, 0 = full extension
[K1] peak flexion measured here = 131.8 deg
[K1] published (Miyashiro & Nagahara 2019, n=79, 40-50 m of a 60 m sprint):
       minimum knee angle in swing 31.6 +/- 5.6 deg on a 180=straight scale
       -> equivalent peak flexion 148.4 deg; ours reads 16.6 deg lower


## [K2] Four summaries of the same curve — it is ROM, not a mean

In [10]:
print(f"[K2] {'summary':22}{'mean':>9}{'r with velocity':>18}{'p':>9}")
for lab, x in [("ROM (max - min)", np.ptp(K,1)), ("max (peak flexion)", K.max(1)),
               ("mean over cycle", K.mean(1)), ("min (nearest straight)", K.min(1))]:
    r_, p_ = st.pearsonr(x, v)
    print(f"     {lab:22}{x.mean():9.1f}{r_:+18.3f}{p_:9.4f}")
print(f"\n[K2] r(ROM, max) = {np.corrcoef(np.ptp(K,1), K.max(1))[0,1]:+.3f}   "
      f"r(ROM, min) = {np.corrcoef(np.ptp(K,1), K.min(1))[0,1]:+.3f}")
print("[K2] -> ROM is driven by peak flexion, not by how straight the leg gets")

[K2] summary                    mean   r with velocity        p
     ROM (max - min)           114.8            -0.552   0.0016
     max (peak flexion)        131.8            -0.506   0.0043
     mean over cycle            66.0            -0.133   0.4837
     min (nearest straight)     16.9            +0.096   0.6133

[K2] r(ROM, max) = +0.741   r(ROM, min) = -0.413
[K2] -> ROM is driven by peak flexion, not by how straight the leg gets


## [K3] The same finding under four constructions

In [11]:
frames = M.frames.values
print(f"[K3] r(stride frames, velocity) = {st.pearsonr(frames, v)[0]:+.3f}  "
      "(faster athletes have shorter strides, so they are interpolated more)")

raw_rom = {}
for f_ in sorted(SP.C3D_DIR.glob("*.c3d")):
    pid = f_.stem.split("-")[0].strip()
    if pid not in cyc: continue
    tr = SP.clean_for_angles(f_); ang, _ = SP.compute_joint_angles(tr["raw64"])
    w, _ = SP.find_top_speed_strides(tr["mk"], tr["peak_frame"])
    raw_rom[pid] = np.mean([np.ptp(ang[a:b+1, N.index("knee_R")]) for a, b in w])
rr = np.array([raw_rom[p] for p in pids])
sm = np.array([stance[p] for p in pids]); nsw = 101 - sm.sum(1)
sw = np.array([np.ptp(K[i][~sm[i]]) for i in range(len(pids))])

print(f"\n[K3] {'construction':44}{'r':>9}{'p':>9}")
for lab, r_, p_ in [
    ("ROM on the 101-point resampled cycle", *st.pearsonr(np.ptp(K,1), v)),
    ("ROM on raw frames, no resampling",     *st.pearsonr(rr, v)),
    ("partial, controlling stride duration", *partial(rr, v, frames)),
    ("swing only, window length controlled", *partial(sw, v, nsw.astype(float))),
]:
    print(f"     {lab:44}{r_:+9.3f}{p_:9.4f}")

[K3] r(stride frames, velocity) = -0.617  (faster athletes have shorter strides, so they are interpolated more)



[K3] construction                                        r        p
     ROM on the 101-point resampled cycle           -0.552   0.0016
     ROM on raw frames, no resampling               -0.567   0.0011
     partial, controlling stride duration           -0.494   0.0055
     swing only, window length controlled           -0.552   0.0016


---
# S · Segmentation, and the artefact

## [S1] The velocity-curve inflection, two methods

In [12]:
print(f"[S1] Kneedle: {M.elbow_t.mean():.2f} s (SD {M.elbow_t.std():.2f}), landing at "
      f"{M.elbow_pct.mean():.1f}% of peak velocity (SD {M.elbow_pct.std():.1f}), n={len(M)}")

[S1] Kneedle: 2.45 s (SD 0.20), landing at 91.0% of peak velocity (SD 2.3), n=30


## [S2] The stance-ROM artefact — window length is set by the outcome

In [13]:
npts = sm.sum(1)
print(f"[S2] stance window spans {npts.min()}-{npts.max()} of 101 points (mean {npts.mean():.0f})")
print(f"[S2] r(stance points, contact time) = {st.pearsonr(npts, gct)[0]:+.3f}  (by construction)")
print(f"[S2] r(stance points, velocity)     = {st.pearsonr(npts, v)[0]:+.3f}")
print(f"\n[S2] {'angle':12}{'r(stROM,v)':>12}{'r(stROM,npts)':>15}{'partial|npts':>14}{'p':>9}")
for j in ["hip_R","hip_L","knee_R","knee_L","ankle_R","shoulder_R","shoulder_L"]:
    Ccol = np.array([cyc[p][:, N.index(j)] for p in pids])
    b_ = np.array([np.ptp(Ccol[i][sm[i]]) for i in range(len(pids))])
    pr, pp = partial(b_, v, npts.astype(float))
    print(f"     {j:12}{st.pearsonr(b_,v)[0]:+12.3f}{st.pearsonr(b_,npts)[0]:+15.3f}"
          f"{pr:+14.3f}{pp:9.4f}")
print("\n[S2] every one loses significance once sample count is controlled")

[S2] stance window spans 4-27 of 101 points (mean 19)
[S2] r(stance points, contact time) = +0.923  (by construction)
[S2] r(stance points, velocity)     = -0.541

[S2] angle         r(stROM,v)  r(stROM,npts)  partial|npts        p
     hip_R             -0.419         +0.773        -0.002   0.9926
     hip_L             -0.481         +0.645        -0.205   0.2766
     knee_R            -0.357         +0.591        -0.055   0.7733
     knee_L            -0.505         +0.588        -0.275   0.1415
     ankle_R           -0.543         +0.648        -0.301   0.1064
     shoulder_R        -0.399         +0.653        -0.073   0.7034
     shoulder_L        -0.378         +0.689        -0.009   0.9624

[S2] every one loses significance once sample count is controlled


## [S3] Gait timing, and the ROM / contact-time trade-off

In [14]:
print(f"[S3] contact {gct.mean()*1000:.0f} ms | flight {M.flight.mean()*1000:.0f} ms | "
      f"stride {M.stride.mean()*1000:.0f} ms | freq {(1/M.stride).mean():.2f} Hz")
for lab, x in [("contact time", gct), ("flight time", M.flight.values),
               ("stride frequency", 1/M.stride.values)]:
    r_, p_ = st.pearsonr(x, v)
    print(f"[S3] r({lab:17}, velocity) = {r_:+.3f}   p = {p_:.5f}")

print(f"\n[S3] {'angle':12}{'r(ROM,v)':>10}{'r(ROM,GCT)':>12}{'partial|GCT':>13}{'beta_ROM':>10}{'beta_GCT':>10}")
gz = (gct - gct.mean()) / gct.std(); vz = (v - v.mean()) / v.std()
for j in ["knee_R", "knee_L", "elbow_R", "hip_R"]:
    a_ = np.ptp(np.array([cyc[p][:, N.index(j)] for p in pids]), 1)
    az = (a_ - a_.mean()) / a_.std()
    B = np.linalg.lstsq(np.column_stack([np.ones(len(v)), az, gz]), vz, rcond=None)[0]
    print(f"     {j:12}{st.pearsonr(a_,v)[0]:+10.3f}{st.pearsonr(a_,gct)[0]:+12.3f}"
          f"{partial(a_,v,gct)[0]:+13.3f}{B[1]:+10.3f}{B[2]:+10.3f}")

[S3] contact 96 ms | flight 369 ms | stride 465 ms | freq 2.16 Hz
[S3] r(contact time     , velocity) = -0.638   p = 0.00015
[S3] r(flight time      , velocity) = -0.041   p = 0.83171
[S3] r(stride frequency , velocity) = +0.617   p = 0.00028

[S3] angle         r(ROM,v)  r(ROM,GCT)  partial|GCT  beta_ROM  beta_GCT
     knee_R          -0.552      +0.240       -0.533    -0.423    -0.536
     knee_L          -0.504      +0.343       -0.395    -0.324    -0.527
     elbow_R         +0.276      -0.436       -0.003    -0.003    -0.639
     hip_R           -0.112      -0.086       -0.218    -0.168    -0.652


---
# M · What the metrics mean

## [M1] ICC(1,1) on the replicate strides

In [15]:
print(f"[M1] {len(PS)} strides from {PS.pid.nunique()} athletes "
      f"({len(PS)/PS.pid.nunique():.1f} each)")
print(f"\n[M1] {'feature':22}{'ICC(1,1)':>10}   reading")
for c in ["trunk_lean_mean","hip_ext_peak","knee_lo_ROM","ankle_lo_mean","knee_hi_mean"]:
    i_ = icc11(PS, c)
    tag = ("excellent" if i_>.9 else "good" if i_>.75 else
           "moderate" if i_>.5 else "POOR - mostly stride noise")
    print(f"     {c:22}{i_:10.3f}   {tag}")
print(f"\n[M1] features with ICC(1,1) > 0.75: "
      f"{sum(icc11(PS,c)>.75 for c in PS.columns if c!='pid')} of {PS.shape[1]-1}")
print(f"[M1] features with ICC(1,1) < 0.50: "
      f"{sum(icc11(PS,c)<.50 for c in PS.columns if c!='pid')}")

[M1] 150 strides from 30 athletes (5.0 each)

[M1] feature                 ICC(1,1)   reading
     trunk_lean_mean            0.889   good
     hip_ext_peak               0.851   good
     knee_lo_ROM                0.859   good
     ankle_lo_mean              0.573   moderate
     knee_hi_mean               0.484   POOR - mostly stride noise

[M1] features with ICC(1,1) > 0.75: 17 of 23
[M1] features with ICC(1,1) < 0.50: 1


---
## How to read these numbers

**ICC(1,1)** — the share of a feature's total variance that is *between athletes*
rather than *between repeated strides of the same athlete*. Here it is a
reliability score: measure the same person five times and see whether you get the
same answer. **0.9** means the feature is essentially a stable property of the
athlete. **0.5** means half of what you measured is stride-to-stride noise, so a
between-athlete correlation using it is attenuated toward zero. **Below 0.5** the
feature is mostly noise and should not enter a between-athlete model at all.
Crucially, ICC never looks at velocity, so screening on it cannot leak.

**A negative r with p < 0.05** — two separate statements. The **sign** says the
direction: negative means as the feature goes up, velocity goes down. The
**p-value** says how often a correlation at least this large would arise from 30
athletes if there were truly no relationship. `r = -0.552, p = 0.0016` therefore
reads: larger knee range of motion goes with slower sprinting, and a correlation
this strong would appear by chance in about 1 run in 600. Neither statement is
about *how much* velocity changes — for that, r-squared says knee ROM accounts
for about 30% of the between-athlete variance in top speed.

**"Collapsing"** — a correlation that disappears once a third variable is held
constant. Stance ROM correlated with velocity at -0.54; but stance *window
length* is set by contact time, which is set by speed, and a longer window
mechanically contains more excursion. Holding sample count fixed, the correlation
falls to -0.30 and stops being significant. The original number was real
arithmetic; it just measured window length rather than biomechanics. A finding
that survives every such control is one where no third variable has been found
that explains it away — never proof of causation, only failure to falsify.